# Projeto: Normalização e Preparação dos Dados Meteorológicos – INPE


## Descrição Geral
Este notebook trata da normalização e preparação dos dados brutos do INPE, extraídos da camada bronze, para análise e consumo pelas próximas camadas (silver/gold).
Antes de serem processados neste notebook, os dados passam por uma etapa automatizada de validação com Great Expectations (GX). Apenas os registros validados com sucesso são utilizados nas transformações abaixo.
As etapas incluem renomeação de colunas, tratamento de tipos de dados, remoção de duplicados, separação de campos compostos e padronização de formatos.

## Objetivos

- Padronizar os nomes das colunas conforme convenção definida.
- Realizar o ajuste de tipos de dados para colunas de datas, timestamps e decimal.
- Realizar a remoção de dados duplicados
- Separar colunas compostas (como precipitação e probabilidade).
- Preparar o DataFrame para persistência na camada silver.

## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake


## Fonte de Dados

- **Localização:** [Blob Storage - previsoes_inpe](grupo01.bronze.inpe)
- **Formato:** Delta
- **Origem:** Tabela bronze criada a partir da ingestão do dataset INPE

---


### Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                            |
|------------|----------------|--------|------------------------------------------------------|
| 16/05/2025 | Evelise Dias   | 1.0    | Pipeline de ingestão do INPE  |
| 16/05/2025 | Evelise Dias   | 1.1    | Criação da tabela individual bronze no medallion     |
| 19/05/2025 | Evelise Dias   | 1.2    | Atualização de documentação     |
| 25/05/2025 | Evelise Dias   | 1.3    | Ajuste do pipeline para ler apenas dados validados pelo GX                     |
| 26/05/2025 | Evelise Dias   | 1.4    | Atualização da documentação para refletir a origem validada dos dados          |


In [0]:
%run ../utils/utils_run_silver 

In [0]:
%run ./queries_silver

In [0]:
# Realiza a ingestão da tabela bronze inpe validada pelo gx
pasta_validos = lakehouse_logs_gx + "/bronze/inpe/validos/inpe"
df_inpe = ler_tabelas(pasta_validos, fromTable=False)

In [0]:
# Salva mapping criado em config
mapping_colunas = CONVENCAO_COLUNAS_INPE

# Renomeia colunas
df_inpe_renomear_colunas = renomear_colunas(df_inpe, mapping_colunas)

In [0]:
## Voltar e ajustar data_carga

# Realiza o ajuste de tipagem de datas
df_inpe_ajustar_tipagem_datas = alterar_tipos(df_inpe_renomear_colunas, ["dt_previsao"], "date")
df_inpe_ajustar_tipagem_datas = alterar_tipos(df_inpe_ajustar_tipagem_datas,["dt_carga"],"timestamp",data_formato="yyyy-MM-dd HH:mm:ss")
# Realiza o ajuste de tipagem de double
df_inpe_ajustar_tipagem_inteiros = alterar_tipos(df_inpe_ajustar_tipagem_datas, ["temp_max", "temp_min"], "decimal")

In [0]:
df_inpe_separar_coluna = tratar_celulas(df_inpe_ajustar_tipagem_inteiros)

In [0]:
df_final = remover_duplicatas(df_inpe_separar_coluna, ["dt_carga", "dt_previsao", "sig_uf","nom_municipio", "flg_fonte_previsao"])

In [0]:
df_com_id = gera_primary_key(df_final)

In [0]:
spark.sql(create_table_silver_inpe)

In [0]:
inserir_com_merge(df_com_id, caminho_tabela_inpe_silver, ["id"])